# Teacher distillation (Colab)

Generate `FeedbackRecord`-aligned JSONL from a stronger teacher model.

**Colab setup:** Runtime → GPU. Add `HF_TOKEN` in Colab secrets (`userdata`).

Output: upload `train.jsonl` + `manifest.json` to `data/distilled/rada-v1/` in the RADA repo.

In [ ]:
# Colab-only: clone or upload RADA, then pip install -e '.[unsloth]'
import json
import os
from datetime import UTC, datetime
from pathlib import Path
from uuid import uuid4

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass  # local run without Colab secrets

from rada.data.cards.feedback_record import (
    FeedbackRecord,
    FeedbackLabels,
    FeedbackProvenance,
)
from rada.models import load_model_registry, resolve_model_path
from rada.backends import StubLLMBackend  # replace with real teacher backend on GPU

TEACHER_ID = "qwen2.5-7b"
CORPUS_NAME = "rada-v1"
OUTPUT_DIR = Path("distilled_output") / CORPUS_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

registry = load_model_registry()
teacher_entry = registry.get_model(TEACHER_ID)
print(f"Teacher: {teacher_entry.hub_path}")

In [ ]:
# Synthetic scenarios — replace with exported reflection JSONL in production runs
SCENARIOS = [
    {"symbol": "BTCUSD", "context": "price rising, low volatility"},
    {"symbol": "ETHUSD", "context": "price falling, high volume"},
    {"symbol": "SOLUSD", "context": "sideways, elevated tail risk"},
]

teacher = StubLLMBackend(model_id=TEACHER_ID)
records = []
for scenario in SCENARIOS:
    prompt = f"Decision trace for {scenario['symbol']}: {scenario['context']}"
    import asyncio
    completion = asyncio.run(teacher.complete(prompt))
    record = FeedbackRecord(
        source="auditor",
        target_project="rada",
        target_card="PreferencePair",
        label_schema="outcome_match",
        payload={"symbol": scenario["symbol"], "teacher_rationale": completion.text},
        provenance=FeedbackProvenance(decision_id=str(uuid4())),
        labels=FeedbackLabels(score=0.92, actual="HOLD"),
        timestamp=datetime.now(tz=UTC),
    )
    records.append(record)

train_path = OUTPUT_DIR / "train.jsonl"
with train_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(record.model_dump_json() + "\n")
print(f"Wrote {len(records)} rows to {train_path}")

In [ ]:
manifest = {
    "name": CORPUS_NAME,
    "teacher_model_id": TEACHER_ID,
    "created_at": datetime.now(tz=UTC).date().isoformat(),
    "row_count": len(records),
    "schema_version": "feedback_record_v1",
    "source": "colab_distill_teacher",
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest

## Train from distilled corpus

```bash
python scripts/reflection_train.py \
  --backend stub \
  --model-id qwen2.5-3b \
  --data-source distilled \
  --distilled-name rada-v1
```